In [ ]:
### Plot entire spectra and corresponding Cyclic voltammetry

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob
import os
import re

# ==========================================
# 1. CONFIGURATION
# ==========================================
raman_folder = 'spectra_Anod-SS1'
cv_file_path = 'cv_data_SS_OER.txt' # Change to your actual CV filename
raman_offset = 700           # Adjust vertical spacing for Raman stack
comment_mark = '#'           # Character at the start of header lines

# Natural sorting for files (ensures 005vr comes before 010vr)
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

files = sorted(glob.glob(os.path.join(raman_folder, "*.txt")), key=natural_sort_key)

# ==========================================
# 2. SETUP PLOT
# ==========================================
fig, (ax_raman, ax_cv) = plt.subplots(1, 2, figsize=(15, 8), 
                                       gridspec_kw={'width_ratios': [2, 1]})

# --- LEFT: Raman Stack ---
colors = plt.cm.viridis(np.linspace(0, 0.9, len(files)))

for i, file_path in enumerate(files):
    # Load Raman: Col 0 = Wavenumber, Col 1 = Intensity
    # sep='\s+' handles spaces or tabs
    raman_data = pd.read_csv(file_path, sep='\s+', comment=comment_mark, header=None, encoding='latin1')
    
    if not raman_data.empty:
        wn = raman_data.iloc[:, 0]
        inty = raman_data.iloc[:, 1]
        label = os.path.basename(file_path).replace('.txt', '')
        
        ax_raman.plot(wn, inty + (i * raman_offset), color=colors[i], label=label, lw=1.2)

ax_raman.set_xlim(60, 4000)
ax_raman.set_xlabel('Wavenumber ($cm^{-1}$)', fontsize=12, fontweight='bold')
ax_raman.set_ylabel('Intensity (a.u.)', fontsize=12, fontweight='bold')
#ax_raman.set_title('Stacked Raman Spectra', fontsize=14, fontweight='bold')
#ax_raman.legend(loc='upper center', fontsize='x-small', ncol=2, frameon=False)

# --- RIGHT: CV Profile (Time on Y-axis) ---
try:
    # 1. Load CV: Skip the header line, auto-detect tabs/spaces
    cv_data = pd.read_csv(
        cv_file_path, 
        sep=None,           # Guesses if it's tab or space
        skiprows=1,         # Skips "E(V) i(A/cm2) T(s)"
        header=None, 
        encoding='latin1',
        engine='python'
    )
    
    # 2. Convert to numeric and remove any empty/text rows
    cv_data = cv_data.apply(pd.to_numeric, errors='coerce').dropna()

    # Mapping: Col 0 = E(V), Col 1 = i(A/cm2), Col 2 = T(s)
    voltage = cv_data.iloc[:, 0]
    current = cv_data.iloc[:, 1]
    time = cv_data.iloc[:, 2]

    # Plot Voltage (X) vs Time (Y)
    ax_cv.plot(voltage, time, color='blue', label='Voltage', lw=1.5)
    ax_cv.set_xlabel('Voltage (V)', color='blue', fontsize=12, fontweight='bold')
    ax_cv.set_ylabel('Time (s)', fontsize=12, fontweight='bold')
    ax_cv.tick_params(axis='x', labelcolor='blue')
    
    # Plot Current (Top X) vs Time (Y)
    ax_cv_twin = ax_cv.twiny()
    ax_cv_twin.plot(current, time, color='red', linestyle='--', alpha=0.7, label='Current')
    ax_cv_twin.set_xlabel('Current ($A/cm^2$)', color='red', fontsize=12, fontweight='bold')
    ax_cv_twin.tick_params(axis='x', labelcolor='red')
    
    #ax_cv.set_title('Electrochemical Profile', fontsize=14, fontweight='bold', pad=20)
    ax_cv.grid(True, linestyle=':', alpha=0.6)

except Exception as e:
    print(f"Error processing CV file: {e}")

plt.tight_layout()
plt.show()

In [ ]:
### PLot for specific window

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob
import os
import re

# ==========================================
# 1. CONFIGURATION
# ==========================================
raman_folder = 'spectra_NiP_HER_1'
cv_file_path = 'CV HER_NiP.txt' # Change to your actual CV filename
raman_offset = 250           # Adjust vertical spacing for Raman stack
comment_mark = '#'           # Character at the start of header lines

# Natural sorting for files (ensures 005vr comes before 010vr)
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

files = sorted(glob.glob(os.path.join(raman_folder, "*.txt")), key=natural_sort_key)

# ==========================================
# 2. SETUP PLOT
# ==========================================
fig, (ax_raman, ax_cv) = plt.subplots(1, 2, figsize=(15, 8), 
                                       gridspec_kw={'width_ratios': [2, 1]})

# --- LEFT: Raman Stack ---
colors = plt.cm.viridis(np.linspace(0, 0.9, len(files)))
for i, file_path in enumerate(files):
    try:
        raman_data = pd.read_csv(file_path, sep='\s+', comment=comment_mark, header=None, encoding='latin1')
        
        if not raman_data.empty:
            wn = raman_data.iloc[:, 0]
            inty = raman_data.iloc[:, 1]
            
            # --- Added Filter for the 350-1000 range ---
            mask = (wn >= 380) & (wn <= 1400)
            wn_filtered = wn[mask]
            inty_filtered = inty[mask]
            
            label = os.path.basename(file_path).replace('.txt', '')
            
            # Plot only the filtered range
            ax_raman.plot(wn_filtered, inty_filtered + (i * raman_offset), 
                          color=colors[i], label=label, lw=1.2)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
ax_raman.set_xlim(380, 1400)
ax_raman.set_xlabel('Wavenumber ($cm^{-1}$)', fontsize=12, fontweight='bold')
ax_raman.set_ylabel('Intensity (a.u.)', fontsize=12, fontweight='bold')
ax_raman.set_title('Stacked Raman Spectra', fontsize=14, fontweight='bold')
ax_raman.legend(loc='upper right', fontsize='x-small', ncol=2, frameon=False)

# --- RIGHT: CV Profile (Time on Y-axis) ---
try:
    # 1. Load CV: Skip the header line, auto-detect tabs/spaces
    cv_data = pd.read_csv(
        cv_file_path, 
        sep=None,           # Guesses if it's tab or space
        skiprows=1,         # Skips "E(V) i(A/cm2) T(s)"
        header=None, 
        encoding='latin1',
        engine='python'
    )
    
    # 2. Convert to numeric and remove any empty/text rows
    cv_data = cv_data.apply(pd.to_numeric, errors='coerce').dropna()

    # Mapping: Col 0 = E(V), Col 1 = i(A/cm2), Col 2 = T(s)
    voltage = cv_data.iloc[:, 0]
    current = cv_data.iloc[:, 1]
    time = cv_data.iloc[:, 2]

    # Plot Voltage (X) vs Time (Y)
    ax_cv.plot(voltage, time, color='blue', label='Voltage', lw=1.5)
    ax_cv.set_xlabel('Voltage (V)', color='blue', fontsize=12, fontweight='bold')
    ax_cv.set_ylabel('Time (s)', fontsize=12, fontweight='bold')
    ax_cv.tick_params(axis='x', labelcolor='blue')
    
    # Plot Current (Top X) vs Time (Y)
    ax_cv_twin = ax_cv.twiny()
    ax_cv_twin.plot(current, time, color='red', linestyle='--', alpha=0.7, label='Current')
    ax_cv_twin.set_xlabel('Current ($A/cm^2$)', color='red', fontsize=12, fontweight='bold')
    ax_cv_twin.tick_params(axis='x', labelcolor='red')
    
    ax_cv.set_title('Electrochemical Profile', fontsize=14, fontweight='bold', pad=20)
    ax_cv.grid(True, linestyle=':', alpha=0.6)

except Exception as e:
    print(f"Error processing CV file: {e}")

plt.tight_layout()
plt.show()

In [ ]:
### Smoothened plot for specific window

import matplotlib.pyplot as plt
from scipy.signal import savgol_filter  
import pandas as pd
import numpy as np
import glob
import os
import re

# ==========================================
# 1. CONFIGURATION
# ==========================================
raman_folder = 'spectra_Anod-SS1'
cv_file_path = 'cv_data_SS_OER.txt' # Change to your actual CV filename
raman_offset = 250           # Adjust vertical spacing for Raman stack
comment_mark = '#'           # Character at the start of header lines

# Natural sorting for files (ensures 005vr comes before 010vr)
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

files = sorted(glob.glob(os.path.join(raman_folder, "*.txt")), key=natural_sort_key)

# ==========================================
# 2. SETUP PLOT
# ==========================================
fig, (ax_raman, ax_cv) = plt.subplots(1, 2, figsize=(8, 8), 
                                       gridspec_kw={'width_ratios': [2, 1]})

# --- LEFT: Raman Stack ---
colors = plt.cm.viridis(np.linspace(0, 0.9, len(files)))
for i, file_path in enumerate(files):
    try:
        raman_data = pd.read_csv(file_path, sep='\s+', comment=comment_mark, header=None, encoding='latin1')
        
        if not raman_data.empty:
            wn = raman_data.iloc[:, 0]
            inty = raman_data.iloc[:, 1]
            
            # --- Added Filter for the 350-1000 range ---
            mask = (wn >= 360) & (wn <= 700)
            wn_filtered = wn[mask]
            inty_filtered = inty[mask]
            inty_smooth = savgol_filter(inty_filtered, window_length=31, polyorder=2)
            label = os.path.basename(file_path).replace('.txt', '')
            
            # Plot only the filtered range
            ax_raman.plot(wn_filtered, inty_smooth + (i * raman_offset), 
                          color=colors[i], label=label, lw=1.2)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
ax_raman.set_xlim(360, 700)
ax_raman.set_xlabel('Wavenumber ($cm^{-1}$)', fontsize=12, fontweight='bold')
ax_raman.set_ylabel('Intensity (a.u.)', fontsize=12, fontweight='bold')
#ax_raman.set_title('Stacked Raman Spectra', fontsize=14, fontweight='bold')
#ax_raman.legend(loc='upper right', fontsize='x-small', ncol=2, frameon=False)

# --- RIGHT: CV Profile (Time on Y-axis) ---
try:
    # 1. Load CV: Skip the header line, auto-detect tabs/spaces
    cv_data = pd.read_csv(
        cv_file_path, 
        sep=None,           # Guesses if it's tab or space
        skiprows=1,         # Skips "E(V) i(A/cm2) T(s)"
        header=None, 
        encoding='latin1',
        engine='python'
    )
    
    # 2. Convert to numeric and remove any empty/text rows
    cv_data = cv_data.apply(pd.to_numeric, errors='coerce').dropna()

    # Mapping: Col 0 = E(V), Col 1 = i(A/cm2), Col 2 = T(s)
    voltage = cv_data.iloc[:, 0]
    current = cv_data.iloc[:, 1]
    time = cv_data.iloc[:, 2]

    # Plot Voltage (X) vs Time (Y)
    ax_cv.plot(voltage, time, color='blue', label='Voltage', lw=1.5)
    ax_cv.set_xlabel('Voltage (V)', color='blue', fontsize=12, fontweight='bold')
    ax_cv.set_ylabel('Time (s)', fontsize=12, fontweight='bold')
    ax_cv.tick_params(axis='x', labelcolor='blue')
    
    # Plot Current (Top X) vs Time (Y)
    ax_cv_twin = ax_cv.twiny()
    ax_cv_twin.plot(current, time, color='red', linestyle='--', alpha=0.7, label='Current')
    ax_cv_twin.set_xlabel('Current ($A/cm^2$)', color='red', fontsize=12, fontweight='bold')
    ax_cv_twin.tick_params(axis='x', labelcolor='red')
    
    #ax_cv.set_title('Electrochemical Profile', fontsize=14, fontweight='bold', pad=20)
    ax_cv.grid(True, linestyle=':', alpha=0.6)

except Exception as e:
    print(f"Error processing CV file: {e}")

plt.tight_layout()
plt.show()